<a href="https://colab.research.google.com/github/41340202s-jpg/ProgramingLanguage/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 主要功能：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- 附加功能：查詢付款人可計算該付款人之總支出與各細項

GoogleSheet: https://docs.google.com/spreadsheets/d/1H_Qe7YSPpavuUY-PM7Z3CZ9icmIYVVlQHQqYGvlE-b4/edit?usp=sharing

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1H_Qe7YSPpavuUY-PM7Z3CZ9icmIYVVlQHQqYGvlE-b4/edit?usp=sharing')

In [ ]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註


In [ ]:
import datetime

In [ ]:
# 讓使用者輸入資料
date_str = input("請輸入日期 (格式：YYYY-MM-DD): ")
# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y-%m-%d')

time_str = input("請輸入時間 (格式：HH:MM): ")
# 檢查時間格式是否正確
datetime.datetime.strptime(time_str, '%H:%M')

item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))
name = input("請輸入付款人: ")
location = input("請輸入地點: ")
pay = input("請輸入支付方式: ")
memo = input("備註(若無請填無): ")

請輸入日期 (格式：YYYY-MM-DD): 2026-03-03
請輸入時間 (格式：HH:MM): 15:04
請輸入品項: 書
請輸入金額: 350
請輸入付款人: k
請輸入地點: 金玉堂
請輸入支付方式: 信用卡
備註(若無請填無): 無


In [ ]:
# 創建一個新的 DataFrame 來代表你要新增的資料
new_data = pd.DataFrame([{
    '日期': date_str,
    '時間': time_str,
    '分類': None,  # 加入 '分類' 欄位並預設為 None
    '品項': item,
    '金額': amount,
    '付款人': name,
    '地點': location,
    '支付方式': pay,
    '備註': memo
}])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [ ]:
# 將 DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
# 替換 NaN 為 None，因為 gspread 不支援 NaN
data_to_write = new_data.replace({float('nan'): None}).values.tolist()

# 第一步：取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 第二步：將資料寫入工作表
worksheet.append_rows(values=data_to_write, value_input_option='USER_ENTERED')
print("資料已成功寫入 Google 工作表！")

資料已成功寫入 Google 工作表！


In [ ]:
# 從 gsheets 的 '工作表1' 載入最新的資料
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 顯示最新的 DataFrame
df

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2026-03-05,9:27,,早餐,50,,,,
1,2026-03-04,16:25,,麵包,50,ay,麵包店,現金,無
2,2026-03-04,16:24,,麵包,50,ay,麵包店,現金,無
3,2026-03-03,15:04,,書,350,k,金玉堂,信用卡,無


In [ ]:
data_to_write[-1]

['2026-03-03', '15:04', None, '書', 350.0, 'k', '金玉堂', '信用卡', '無']

In [ ]:
payer_name = input("請輸入您想查詢的付款人姓名: ")
print(f"您輸入的付款人姓名是: {payer_name}")

請輸入您想查詢的付款人姓名: k
您輸入的付款人姓名是: k


In [ ]:
payer_df = df[df['付款人'] == payer_name]
payer_df.head()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
3,2026-03-03,15:04,,書,350,k,金玉堂,信用卡,無


In [ ]:
payer_df['金額'] = pd.to_numeric(payer_df['金額'])
total_spending = payer_df['金額'].sum()
print(f"Total spending for {payer_name}: {total_spending:.2f}")

Total spending for k: 350.00


/tmp/ipykernel_271/3275995595.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  payer_df['金額'] = pd.to_numeric(payer_df['金額'])


In [ ]:
payer_df.loc[:, '金額'] = pd.to_numeric(payer_df['金額'])
total_spending = payer_df['金額'].sum()
print(f"{payer_name}的付款總金額: {total_spending:.2f}")

k的付款總金額: 350.00
